# Grocery Stock Detection - Full YOLOv8 Training (GPU)\n
\n
Run this notebook on **Google Colab** with a **GPU runtime** (Runtime -> Change runtime type -> T4 GPU).\n
\n
Steps:\n
1. (Optional, on your machine) Create `grocery.v9i.yolov8.zip` from the dataset folder.\n
2. Upload the zip (or connect Google Drive) in the dataset cell.\n
3. Train on the full 62-class dataset (~24k train images) for 25 epochs.\n
4. Validate on the hold-out test set, export to ONNX, and download `best.pt`.\n
\n
Copy the downloaded `best.pt` back to this project's `models/` folder as `models/best.pt` and it will be used automatically by `predict.py`, `monitor.py` and the rest of the stack.

## 0. (Run on YOUR machine, not Colab) Create the dataset zip\n
\n
Only if you haven't already got `grocery.v9i.yolov8.zip`. Leave this cell unchanged on Colab.

In [4]:
import os
import zipfile

folder = "grocery.v9i.yolov8"
zip_file = "grocery.v9i.yolov8.zip"

if os.path.isdir(folder) and not os.path.exists(zip_file):

    with zipfile.ZipFile(zip_file, "w", zipfile.ZIP_DEFLATED) as zf:

        for root, _, files in os.walk(folder):

            for f in files:
                # Skip temporary files
                if f.endswith("~"):
                    continue

                file_path = os.path.join(root, f)

                # Keep the folder structure inside the ZIP
                arcname = os.path.relpath(file_path, folder)

                zf.write(file_path, arcname)

    size_mb = os.path.getsize(zip_file) // (2 ** 20)

    print(
        f"Created {zip_file} successfully — {size_mb} MB"
    )

else:
    print(
        "Skip: folder doesn't exist or ZIP file already exists."
    )

Skip: folder doesn't exist or ZIP file already exists.


In [5]:
!nvidia-smi -L
!python --version

'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


Python 3.14.0


In [6]:
%pip install -q ultralytics

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Load the dataset

Either upload the zip from the local project folder:

In [7]:
from google.colab import files
files.upload()  # select grocery.v9i.yolov8.zip

ModuleNotFoundError: No module named 'google'

Or, if the zip is already in Google Drive, mount Drive and copy it. Skip whichever cell does not apply.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import os
import zipfile

ZIP = 'grocery.v9i.yolov8.zip'
if not os.path.exists(ZIP):
    ZIP = '/content/drive/MyDrive/grocery.v9i.yolov8.zip'

with zipfile.ZipFile(ZIP) as zf:
    zf.extractall('/content/dataset')

base = '/content/dataset'
train = sorted(os.listdir(f'{base}/train/images'))
valid = sorted(os.listdir(f'{base}/valid/images'))
test = sorted(os.listdir(f'{base}/test/images'))
print(f'train={len(train)} valid={len(valid)} test={len(test)}')

In [ ]:
import yaml

with open(f'{base}/data.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['train'] = f'{base}/train/images'
cfg['val'] = f'{base}/valid/images'
cfg['test'] = f'{base}/test/images'
cfg.pop('roboflow', None)

with open(f'{base}/data.yaml', 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(f"classes={cfg['nc']} -> {len(cfg['names'])} names")
print(open(f'{base}/data.yaml').read())

## 2. Train the full model

Tuned hyperparameters: `AdamW`, cosine LR, `close_mosaic` for the last 10 epochs, patience 15, 640px input. 25 epochs on a T4 takes roughly 2-3 hours.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=f'{base}/data.yaml',
    epochs=25,
    imgsz=640,
    batch=16,
    device=0,
    workers=8,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    momentum=0.9,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    cos_lr=True,
    close_mosaic=10,
    patience=15,
    seed=42,
    project='/content/runs',
    name='stock_detector',
    exist_ok=True,
    plots=True,
)

print(results.save_dir)

In [ ]:
# Resume from checkpoint if the session drops
from ultralytics import YOLO
import os

ckpt = '/content/runs/stock_detector/weights/last.pt'
if os.path.exists(ckpt):
    YOLO(ckpt).train(resume=True)

## 3. Evaluate on the hold-out test set

In [ ]:
from ultralytics import YOLO

best = YOLO('/content/runs/stock_detector/weights/best.pt')
metrics = best.val(data=f'{base}/data.yaml', split='test', imgsz=640, batch=16)
print(f"mAP50-95={metrics.box.map:.4f}  mAP50={metrics.box.map50:.4f}  precision={metrics.box.mp:.4f}  recall={metrics.box.mr:.4f}")

## 4. Export & download

In [ ]:
best.export(format='onnx', imgsz=640, simplify=True)  # for Node.js backend\n
print('ONNX exported')

In [ ]:
import glob, shutil, os
from google.colab import files

weights_dir = '/content/runs/stock_detector/weights'
shutil.copy(f'{weights_dir}/best.pt', '/content/best.pt')
files.download('/content/best.pt')

onnx = glob.glob(f'{weights_dir}/best.onnx') or glob.glob('/content/best.onnx')
if onnx:
    files.download(onnx[0])

# After download, on your machine:\n
#   mkdir -p models && cp best.pt models/best.pt\n
#   python predict.py --source <image_or_folder> --model models/best.pt --imgsz 640\n
#   python monitor.py --source 0 --model models/best.pt --api http://localhost:4000